# Calibracao residual DMSO nao incorporada

**Papel nesta entrega:** Suplementar: analise historica descrita no trabalho

**Origem:** https://colab.research.google.com/drive/1jI5O2getUZOFGvQDNC8dCd5wIDyhwq-A

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Dependencias e caminhos sao os da campanha historica, nao das quatro execucoes padronizadas. Esses cadernos nao devem substituir os resultados principais. Consulte o mapa de cobertura e mantenha separadas as datas de execucao. 

# Etapa 07 | Calibracao residual dos contrastes por DMSO

**Pergunta:** descontar a referencia dos DMSO por placa e replica melhora a concordancia dos tratamentos em R1-R3, preservando as diferencas entre condicoes?

Este notebook e independente: nao precisa executar novamente os anteriores. Ele le os arquivos originais configurados, sem os sobrescrever. **R4 nao e lida.**

A etapa 06 (RMS sem P95) permanece como comparacao anterior. Esta e uma nova tentativa, baseada em uma unica correcao de localizacao dos contrastes de A. Nao e uma correcao validada de todo o pipeline.

Os calculos deste notebook foram testados localmente com os arquivos reais. Os resultados locais estao separados no pacote; **as celulas abaixo serao executadas na sua sessao Colab**. Nenhum numero do novo resultado foi preenchido manualmente.


In [ ]:
from pathlib import Path
import os, sys, subprocess, importlib.util
try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
requirements = [('numpy','numpy'), ('pandas','pandas'), ('scipy','scipy'),
                ('sklearn','scikit-learn'), ('matplotlib','matplotlib'),
                ('threadpoolctl','threadpoolctl')]
if IN_COLAB:
    requirements.append(('polars','polars'))
for module, package in requirements:
    if importlib.util.find_spec(module) is None:
        if not IN_COLAB:
            raise ImportError('Instale a dependencia: '+package)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
from IPython.display import display, Markdown, Image
RAIZ = Path(os.environ.get('TCC_RAIZ', '/content/drive/MyDrive/mba_tcc'))
print('Raiz de entrada:', RAIZ)


## Protocolo fixo antes do ajuste

Duas versoes: **A original** e **A com centro dos contrastes DMSO descontado**. Mesmas 2.447 condicoes completas, quatro canais (AGP, DNA, ER, Mito), K-means com **k=4**, 30 inicializacoes e ate 500 iteracoes. Sem busca de hiperparametros, sem alteracao de P95, sem calibracao de variancia/covariancia e sem exclusao de controles.

Por placa p, replica r e canal b:
\[
C_{iprb}=\log(1+A_{iprb})-\frac14\sum_c\log(1+A_{iprc}),\qquad
\mu_{prb}=\operatorname{media}_{DMSO}C_{prb},\qquad C^*_{iprb}=C_{iprb}-\mu_{prb}.
\]

A media dos controles preserva a soma zero do vetor de contrastes. O ajuste principal usa os 28 DMSO por placa x replica. Isso nao torna o centro estimado livre de incerteza: 20 divisoes aleatorias de 14/14 avaliam sua sensibilidade.

**Ordem de agregacao explicita:** para as representacoes de duas ou tres replicas, preservamos `C(mediana de A)` da etapa 04 e descontamos a **media dos centros DMSO das replicas incluidas**. Nao usamos `mediana(C-mu)`, que mudaria o comparador por nao comutar com a transformacao. Para uma replica a formula coincide exatamente com a calibracao por poco. Com correcao zero a entrada original e reproduzida.

Na rotacao R1/R2 -> R3, apenas R1/R2 ajustam escala e K-means; os DMSO de R3 estimam a referencia de aplicacao dessa placa, sem utilizar os tratamentos de R3. A normalizacao MAD, a selecao e o P95 anteriores continuam fixos e ja usaram R1-R3; esta e uma avaliacao retrospectiva **condicional**, nao um novo teste independente.


## O que conta como evidencia

Nao basta centralizar os proprios DMSO em zero. Avaliamos os **14 controles deixados fora** de cada ajuste adicional. As divisoes nao sao independentes e nao reconstituem a separacao das etapas antigas.

Nos tratamentos, comparamos ARI entre replicas, composicao/tamanho dos grupos e recuperacao da **mesma condicao** entre replicas contra outros compostos da mesma placa. As distancias usam a mesma escala (IQR do comparador original ajustado nas duas replicas de desenvolvimento) antes/depois, para nao criar uma melhora artificial alterando a metrica. Empates contam metade; a fracao de pares mais proximos nao e acuracia supervisionada.

A traducao preserva distancias dentro de cada placa e replica antes da escala global. Isso e conferido numericamente, mas nao comprova preservacao de toda biologia. Efeitos de posicao de poco e componentes dependentes do tratamento nao sao separados. Nenhum grupo sera automaticamente denominado mitocondrial; nenhum status dos 12 candidatos sera alterado.


### Fontes fixas e verificacoes

In [ ]:
"""Calibracao residual por DMSO: ensaio metodologico retrospectivo, R1-R3."""
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, shutil
import numpy as np
import pandas as pd
import sklearn
from sklearn.cluster import KMeans
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score, silhouette_score
from threadpoolctl import threadpool_limits
import matplotlib.pyplot as plt

P,R,W,D,E='Metadata_Plate_ID','Metadata_Replicate','Metadata_Well','Metadata_Concentration','Metadata_EOS'
KEY=[P,E,'_dose_key']
CHANNELS=['AGP','DNA','ER','Mito']
BLOCKS=CHANNELS+['Morfologia_Cells','Morfologia_Cyto','Morfologia_Nuc']
REPS=['R1','R2','R3']; PLATES=['B100'+str(i) for i in range(1,8)]
SPACES={'magnitude_4canais':2,'contrastes_4canais':4}
CFG={'seed':20260927,'n_init':30,'max_iter':500,'n_splits':20,'n_calibration':14,'k':4,'aggregation':'C(median(A, replicas)) - mean(mu_DMSO, replicas)','primary_scope':'contrastes_4canais','protocol_version':'07.1'}
SOURCES = {
  "complete": {
    "relative_path": "resultados_tcc/mapa_multibloco/R1_R3_20260926T152153_537184Z/tabelas/completas.csv",
    "sha256": "de4566acaa64208b5e41294ac9e4acc506833a23541ae65f5c0f383fa05bddcf",
    "drive_url": "https://drive.google.com/file/d/1vB1LoU1oZQg1CxjYqKbRClxhfsNblU2P/view"
  },
  "refs": {
    "relative_path": "pipeline_IMTM_HepG2/20260919T204105_625149Z/normalizacao_20260919T205516_579360Z/feature_selection_20260919T211235_656816Z/escores_exploratorios_20260919T230543_380473Z/referencias_dmso.csv",
    "sha256": "05ea6446566459559a376078d14c7ec5ef98821e4d34bab6c788ac8ebed6d952",
    "drive_url": "https://drive.google.com/file/d/1SgytsR_sz-st5I7Q0ive0SewOZmh5yKx/view"
  },
  "dmso": {
    "relative_path": "resultados_tcc/auditoria_DMSO_P95/R1_R3_20260927T123126_778389Z/tabelas/DMSO_escores_e_contrastes.csv",
    "sha256": "4721bd5db4f9a9dd820438d940abbf0ba0622e0e32e7f1977b4f25cf17f9de43",
    "drive_url": "https://drive.google.com/file/d/19CUP_xj1M0Cspe2KkL_lXr7zZ0mdko87/view"
  },
  "models": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/modelos_numericos.json",
    "sha256": "5839e78093939ff2c89c6f329ae41588023ad2121f53f8909905a59100656985",
    "drive_url": "https://drive.google.com/file/d/1GDmrHEMOFWuZPj4Zt57qVU7qqNTKGK63/view"
  },
  "groups": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/tabelas/grupos.csv",
    "sha256": "801a90adf37da01108aec71acae2c9f78c5c3552248faadb61fdf2e3398c6019",
    "drive_url": "https://drive.google.com/file/d/1wf33vAKy6IBYswYG1t7e8HMFkmpdI1eA/view"
  },
  "assign": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/tabelas/atribuicoes.csv",
    "sha256": "27a1aaff1046de55e0721f62e1cfe166c83300df7192785e02e5616c893c43c4",
    "drive_url": "https://drive.google.com/file/d/19-cPZoxd0JTW6jOLP_IZ4Md9cVsWJZys/view"
  },
  "candidates": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/tabelas/candidatos.csv",
    "sha256": "4717be42bbf400925149d07339036735474575ec37e4b766ee5b8dc5a0df8ca7",
    "drive_url": "https://drive.google.com/file/d/1zhBCydnXSkOH2lMU2tqCczXr1_BfslDn/view"
  },
  "subs": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/tabelas/subamostras.csv",
    "sha256": "43b932704416ca8cbe0afa1387bdb68813492a8e8fcf3a421afbc987d6c4e46f",
    "drive_url": "https://drive.google.com/file/d/1Kjzgx_M1hibP8FRO4U6ybONxfvJnKd0e/view"
  },
  "rot": {
    "relative_path": "resultados_tcc/agrupamento_nao_supervisionado/R1_R3_20260926T154038_658280Z/tabelas/rotacoes.csv",
    "sha256": "d5a9cb08b0826a08588c7c5d7c5cd3c4ac7b225d0009feaadecda9e0360a55ee",
    "drive_url": "https://drive.google.com/file/d/145RsDiUYLFZey92LdmOBe8MJh489kWka/view"
  }
}


def check(ok,message):
    if not bool(ok): raise ValueError(message)


def close(a,b,message):
    check(np.allclose(a,b,rtol=1e-9,atol=1e-10,equal_nan=False),message)


def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()


def read_csv(path):
    """Polars quando disponivel; pandas como alternativa explicita para CSV."""
    try: import polars as pl
    except ImportError: return pd.read_csv(path,dtype={D:str}),'pandas'
    import csv
    with Path(path).open(encoding='utf-8-sig',newline='') as f: header=next(csv.reader(f))
    overrides={D:pl.String} if D in header else None
    z=pl.read_csv(path,infer_schema_length=10000,schema_overrides=overrides)
    return pd.DataFrame(z.to_dict(as_series=False)),'polars'


def bool_col(s):
    z=s.astype(str).str.lower().str.strip().map({'true':True,'false':False,'1':True,'0':False})
    check(z.notna().all(),'Booleano invalido.');return z.astype(bool)


def rep(x,space):
    x=np.asarray(x,float)
    check(x.ndim==2 and x.shape[1]==4 and np.isfinite(x).all() and (x>=0).all(),
          'Representacao exige quatro canais finitos e nao negativos.')
    l=np.log1p(x)
    if space=='contrastes_4canais': return l-l.mean(axis=1,keepdims=True)
    check(space=='magnitude_4canais','Espaco desconhecido.');return l


def fixed_predict(x,m):
    base=rep(x,m['espaco']);sc=np.asarray(m['scale'],float)
    check(np.all(sc>0) and m['channels']==CHANNELS,'Modelo salvo invalido.')
    z=(base-np.asarray(m['center']))/sc
    d=((z[:,None,:]-np.asarray(m['cluster_centers'])[None,:,:])**2).sum(axis=2)
    return d.argmin(axis=1),z


def load(root):
    frames={};paths={};readers={}
    for name,s in SOURCES.items():
        p=Path(root)/s['relative_path'];check(p.is_file(),'Fonte ausente: '+str(p))
        check(sha256(p)==s['sha256'],'Integridade alterada: '+str(p))
        paths[name]=p
        if name=='models':frames[name]=json.loads(p.read_text())
        else:frames[name],readers[name]=read_csv(p)
    df=frames['complete'].copy();at=frames['assign'].copy();dm=frames['dmso'].copy()
    for z in (df,at):
        check(not z[KEY].isna().any().any(),'Chave nula.')
        for c in KEY:z[c]=z[c].astype(str).str.strip();check(z[c].ne('').all(),'Chave vazia.')
        check(not z.duplicated(KEY).any() and len(z)==2447,'Condicoes duplicadas ou numero diferente.')
        check(not any('R4' in c for c in z),'Fonte contem coluna R4.')
    df=df.sort_values(KEY,kind='stable').reset_index(drop=True)
    at=at.sort_values(KEY,kind='stable').reset_index(drop=True)
    check(df[KEY].equals(at[KEY]),'Chaves das atribuicoes divergem.')
    check(bool_col(df.completa_3rep_7blocos).all(),'Condicoes incompletas; nao ha imputacao.')
    df['prioridade_R1_R3_original']=bool_col(df.prioridade_R1_R3_original)
    check(df.prioridade_R1_R3_original.sum()==12,'Prioridades originais divergiram.')
    check(set(df[P])==set(PLATES),'Placas diferentes.')
    refs=frames['refs'].query('bloco in @BLOCKS').copy()
    check(len(refs)==49 and not refs.duplicated([P,'bloco']).any(),'Referencias ausentes/duplicadas.')
    check((refs.n_dmso_validos==84).all() and np.isclose(refs.quantil,.95).all(),'P95 com amostragem diferente.')
    q=refs.pivot(index=P,columns='bloco',values='limiar_RMS').loc[PLATES,BLOCKS]
    check(np.isfinite(q).all().all() and (q>0).all().all(),'P95 nao finito/positivo.')
    a=np.stack([df[['A_'+b+'_'+r for b in BLOCKS]].to_numpy(float) for r in REPS],axis=1)
    check(np.isfinite(a).all() and (a>=0).all(),'Escores nao finitos/negativos.')
    close(np.median(a,axis=1),df[['A_'+b+'_mediana' for b in BLOCKS]],'Medianas nao reproduzidas.')
    # P95 e constante por placa/bloco: mediana(A*P95)=mediana(A)*P95.
    # A auditoria 05 conferiu esta identidade nos arquivos de pocos originais.
    qrows=q.loc[df[P],BLOCKS].to_numpy(float)
    ecube=a*qrows[:,None,:]
    close(ecube/qrows[:,None,:],a,'Falha na identidade E=A*P95.')
    check(len(dm)==588 and not dm.duplicated([P,R,W]).any(),'DMSO duplicados ou numero incorreto.')
    check(set(dm[R])==set(REPS) and set(dm[P])==set(PLATES),'DMSO fora do escopo.')
    counts=dm.groupby([P,R]).size()
    check(len(counts)==21 and (counts==28).all(),'DMSO: esperados 28 por placa e replica.')
    av=dm[['A_'+b for b in CHANNELS]].to_numpy(float)
    ev=dm[['E_'+b for b in CHANNELS]].to_numpy(float)
    close(ev/q.loc[dm[P],CHANNELS].to_numpy(),av,'DMSO: A diferente de E/P95.')
    check(np.isfinite(ev).all() and (ev>=0).all(),'RMS dos controles invalido.')
    for pref,raw in [('C_A_',av),('C_E_',ev)]:
        close(rep(raw,'contrastes_4canais'),dm[[pref+b for b in CHANNELS]],'Contrastes DMSO diferentes.')
    frozen={};maps={};conferences=[]
    for space,k in SPACES.items():
        gs=frames['groups'].query('espaco == @space');check(set(gs.k)=={k},'k original diferente.')
        mm=[m for m in frames['models'] if m['espaco']==space and m['k']==k]
        check(len(mm)==1,'Modelo original ausente/ambiguo.');m=mm[0]
        mapping=dict(zip(gs.id_modelo.astype(int),gs.grupo.astype(str)))
        for r,raw in [('mediana',np.median(a[:,:,:4],axis=1))]+[(r,a[:,i,:4]) for i,r in enumerate(REPS)]:
            y,_=fixed_predict(raw,m);lab=np.array([mapping[int(g)] for g in y])
            col=space+('_grupo' if r=='mediana' else '_'+r)
            check(np.array_equal(lab,at[col]),'Modelo original nao reproduz '+col)
            conferences.append({'espaco':space,'projecao':r,'n':len(y),'n_divergencias':0})
        check(len(frames['rot'].query('espaco == @space and k == @k'))==3,'Rotacoes originais ausentes.')
        check(len(frames['subs'].query('espaco == @space and k == @k'))==20,'Subamostras originais ausentes.')
        frozen[space]=m;maps[space]=mapping
    return {'df':df,'A':a,'E':ecube,'dmso':dm,'frozen':frozen,'maps':maps,'frames':frames,
            'paths':paths,'readers':readers,'conferences':pd.DataFrame(conferences)}

### Calibracao e avaliacao dos controles

In [ ]:
# --- Experimental protocol: a single location correction, no parameter search. ---
from scipy.spatial.distance import cdist, pdist
SPACE='contrastes_4canais'
VARIANTS=['A_original','A_centrado_DMSO']
LABELS={'A_original':'Original','A_centrado_DMSO':'Calibrado por DMSO'}


def fit_contrasts(x,seed):
    """Fit only the robust scaler and K-means; input may be signed contrasts."""
    x=np.asarray(x,float)
    check(x.ndim==2 and x.shape[1]==4 and np.isfinite(x).all(),'Contrastes invalidos.')
    check(np.unique(x,axis=0).shape[0]>=CFG['k'],'Perfis distintos insuficientes.')
    scaler=RobustScaler(quantile_range=(25,75)).fit(x)
    z=scaler.transform(x)
    km=KMeans(n_clusters=CFG['k'],n_init=CFG['n_init'],max_iter=CFG['max_iter'],
              init='k-means++',random_state=int(seed),algorithm='lloyd').fit(z)
    check(len(np.unique(km.labels_))==CFG['k'],'Agrupamento degenerado; nao eliminar grupos silenciosamente.')
    return {'channels':CHANNELS,'k':CFG['k'],'center':scaler.center_.tolist(),
            'scale':scaler.scale_.tolist(),'cluster_centers':km.cluster_centers_.tolist(),
            'seed':int(seed)}, km.labels_.copy(),z


def predict_contrasts(x,m):
    z=(np.asarray(x,float)-np.array(m['center']))/np.array(m['scale'])
    d=cdist(z,np.array(m['cluster_centers']),metric='sqeuclidean')
    return d.argmin(1),z


def control_cube(dm):
    """Stable ordering: seven plates x three replicates x 28 wells x four channels."""
    v=np.empty((7,3,28,4));w=np.empty((7,3,28),dtype=object)
    for ip,p in enumerate(PLATES):
        for ir,r in enumerate(REPS):
            part=dm.loc[(dm[P]==p)&(dm[R]==r)].sort_values(W,kind='stable')
            check(len(part)==28 and part[W].is_unique,'DMSO ausentes/duplicados.')
            v[ip,ir]=rep(part[['A_'+b for b in CHANNELS]].to_numpy(float),SPACE)
            w[ip,ir]=part[W].astype(str).to_numpy()
    close(v.sum(-1),np.zeros((7,3,28)),'Contrastes nao somam zero.')
    return v,w


def center_controls(dc,selection=None):
    """Estimate the center exclusively from DMSO; no treatment argument is accepted."""
    if selection is None: return dc.mean(axis=2)
    check(selection.shape==dc.shape[:3] and selection.dtype==bool,'Selecao de controles invalida.')
    n=selection.sum(2)
    check((n==CFG['n_calibration']).all(),'Esperados 14 controles para cada centro.')
    return np.sum(dc*selection[:,:,:,None],axis=2)/n[:,:,None]


def split_controls(seed):
    rng=np.random.default_rng(seed);sel=np.zeros((7,3,28),bool)
    for ip in range(7):
        for ir in range(3):sel[ip,ir,rng.permutation(28)[:CFG['n_calibration']]]=True
    return sel


def aggregate_profile(acube,plate_index,rep_indices,mu=None):
    """Keep the original aggregation: C(median A). Then subtract the average
    DMSO contrast center of ONLY the included replicates. This is explicitly
    not median(C - mu), since centering and nonlinear aggregation do not commute.
    Single-replicate use is exactly C_pr - mu_pr. With mu=0 the old baseline is exact.
    """
    ii=list(rep_indices)
    check(ii and set(ii).issubset({0,1,2}),'Replicas invalidas/R4 proibida.')
    x=rep(np.median(acube[:,ii,:4],axis=1),SPACE)
    if mu is not None:x=x-mu[plate_index][:,ii,:].mean(axis=1)
    close(x.sum(1),np.zeros(len(x)),'Perda da restricao de soma zero.')
    return x


def paired_retrieval(train,test,df,scale):
    """Same-condition retrieval against other compounds in the SAME plate.
    Use a common metric: baseline IQR learned on the two training replicates.
    No per-variant metric fit is used for this comparison. Half credit for ties.
    Identity is experimental condition, not biological ground truth; no p-values.
    """
    rows=[]
    for plate in PLATES:
        ids=np.flatnonzero(df[P].to_numpy()==plate)
        compound=df.iloc[ids][E].to_numpy()
        d=cdist(train[ids]/scale,test[ids]/scale,metric='euclidean')
        paired=np.diag(d)
        eligible=compound[:,None]!=compound[None,:]
        for i,idx in enumerate(ids):
            neg=d[i,eligible[i]];check(len(neg)>0,'Sem outros compostos para comparacao.')
            atol=1e-12*max(1.,paired[i]);ties=np.abs(neg-paired[i])<=atol
            better=(neg>paired[i]+atol).sum()
            rank=1+(neg<paired[i]-atol).sum()+.5*ties.sum()
            rows.append({'condition_index':int(idx),'placa':plate,'distancia_par':float(paired[i]),
                         'distancia_outros_mediana':float(np.median(neg)),
                         'fracao_par_mais_proximo':float((better+.5*ties.sum())/len(neg)),
                         'reciproco_rank':float(1/rank),'n_compostos_comparados':int(len(neg))})
    return pd.DataFrame(rows).sort_values('condition_index').reset_index(drop=True)


def retrieval_summary(t):
    # Equal weight to conditions, not to all pair combinations.
    return {'distancia_par_mediana':float(t.distancia_par.median()),
            'distancia_outros_mediana':float(t.distancia_outros_mediana.median()),
            'fracao_par_mais_proximo_media':float(t.fracao_par_mais_proximo.mean()),
            'fracao_par_mais_proximo_mediana':float(t.fracao_par_mais_proximo.median()),
            'reciproco_rank_medio':float(t.reciproco_rank.mean()),
            'razao_distancias_medianas':float(t.distancia_par.median()/t.distancia_outros_mediana.median())}


def evaluate_rotation(inp,mu,ir,seed,with_retrieval=False):
    idx=inp['plate_index'];included=[j for j in range(3) if j!=ir]
    xt=aggregate_profile(inp['A'],idx,included,mu)
    xv=aggregate_profile(inp['A'],idx,[ir],mu)
    model,yt,_=fit_contrasts(xt,seed);yv,_=predict_contrasts(xv,model)
    row={'replica_aplicacao':REPS[ir],'replicas_ajuste':'+'.join(REPS[j] for j in included),
         'ARI':float(adjusted_rand_score(yt,yv)),'pct_mesmo_grupo':float(100*np.mean(yt==yv)),
         'n_grupos_aplicacao':int(len(np.unique(yv))),
         'menor_grupo_ajuste':int(np.bincount(yt,minlength=4).min())}
    tab=None
    if with_retrieval:
        tab=paired_retrieval(xt,xv,inp['df'],inp['common_scales'][ir])
        row.update(retrieval_summary(tab))
    return row,tab,model


def heldout_controls(dc,selection,mu):
    """Return diagnostics on the complementary 14 wells, never the calibration mean."""
    leftover=~selection
    n=leftover.sum(2)
    check((n==14).all() and not np.any(leftover&selection),'Separacao dos controles incorreta.')
    mean_before=np.sum(dc*leftover[:,:,:,None],axis=2)/n[:,:,None]
    mean_after=mean_before-mu
    out=[]
    for name,means in [('A_original',mean_before),('A_centrado_DMSO',mean_after)]:
        flat=means.reshape(21,4)
        dispersion=np.sqrt(np.mean((flat-flat.mean(0))**2))
        bias=np.sqrt(np.mean(flat**2))
        for ip,p in enumerate(PLATES):
            for ir,r in enumerate(REPS):
                for j,b in enumerate(CHANNELS):
                    out.append({'variante':name,'placa':p,'replica':r,'canal':b,
                                'media_controles_avaliacao':float(means[ip,ir,j]),'n_avaliacao':14,
                                'dispersao_medias_21_grupos':float(dispersion),
                                'RMS_medias_em_relacao_zero':float(bias)})
    return out

### Comparacao nos tratamentos

In [ ]:
def analyse_calibration(inp,out):
    df=inp['df'];dc,wells=control_cube(inp['dmso']);mu=center_controls(dc)
    idx=np.array([PLATES.index(p) for p in df[P]],dtype=int);inp['plate_index']=idx
    m0=inp['frozen'][SPACE];seed=int(m0['seed']);check(m0['k']==4,'Baseline nao usa k=4.')
    inp['common_scales']={}
    for r in range(3):
        x=aggregate_profile(inp['A'],idx,[j for j in range(3) if j!=r])
        inp['common_scales'][r]=RobustScaler().fit(x).scale_
    assign=df[KEY+[D,'prioridade_R1_R3_original']].copy()
    summaries=[];rotation_rows=[];retrieval_rows=[];perplate_rows=[];profile_rows=[];models={}
    murows=[]
    for ip,p in enumerate(PLATES):
        for ir,r in enumerate(REPS):
            for j,b in enumerate(CHANNELS):
                murows.append({'placa':p,'replica':r,'canal':b,'centro_DMSO':float(mu[ip,ir,j]),'n':28})
    with threadpool_limits(limits=1):
        for variant,centering in [('A_original',None),('A_centrado_DMSO',mu)]:
            x=aggregate_profile(inp['A'],idx,[0,1,2],centering)
            if centering is None:
                model=m0;y,z=predict_contrasts(x,model);mapping=inp['maps'][SPACE]
                # Refit as an execution check, not another optimized baseline.
                mf,yf,_=fit_contrasts(x,seed)
                close(adjusted_rand_score(y,yf),1.,'Ajuste original nao reproduz particao; conferir versoes.')
            else:
                model,y,z=fit_contrasts(x,seed)
                magnitude=np.log1p(np.median(inp['A'][:,:,:4],axis=1)).mean(1)
                order=sorted(set(y),key=lambda g:(np.median(magnitude[y==g]),int(g)))
                mapping={int(g):'J'+str(i+1).zfill(2) for i,g in enumerate(order)}
            rr=np.column_stack([predict_contrasts(aggregate_profile(inp['A'],idx,[r],centering),model)[0] for r in range(3)])
            stable=(rr==rr[:,[0]]).all(1)
            assign[variant+'_grupo']=[mapping[int(g)] for g in y]
            assign[variant+'_3rep_mesmo_grupo']=stable
            for ir,r in enumerate(REPS):assign[variant+'_'+r]=[mapping[int(g)] for g in rr[:,ir]]
            models[variant]=dict(model,label_map=mapping,correcao=variant,aggregation=CFG['aggregation'])
            for ir in range(3):
                row,detail,_=evaluate_rotation(inp,centering,ir,seed+20000+ir,True)
                row['variante']=variant;rotation_rows.append(row)
                detail['variante']=variant;detail['replica_aplicacao']=REPS[ir];retrieval_rows.append(detail)
                if centering is None:
                    expected=inp['frames']['rot'].loc[(inp['frames']['rot'].espaco==SPACE)&(inp['frames']['rot'].k==4)&(inp['frames']['rot'].replica_aplicacao==REPS[ir])]
                    check(len(expected)==1,'Rotacao original ausente.')
                    close(row['ARI'],expected.ARI.iloc[0],'ARI original da rotacao nao reproduzido.')
            rt=pd.DataFrame(rotation_rows).query('variante==@variant')
            summaries.append({'variante':variant,'n_condicoes':len(df),'k':4,
                 'silhouette_interno':float(silhouette_score(z,y)),
                 'AMI_placa':float(adjusted_mutual_info_score(df[P],y)),
                 'ARI_rotacoes_mediana':float(rt.ARI.median()),
                 'ARI_rotacoes_min':float(rt.ARI.min()),'ARI_rotacoes_max':float(rt.ARI.max()),
                 'pct_3rep_mesmo_grupo':float(100*stable.mean()),
                 'fracao_par_mais_proximo_media_3rot':float(rt.fracao_par_mais_proximo_media.mean()),
                 'distancia_par_mediana_3rot':float(rt.distancia_par_mediana.median()),
                 'distancia_outros_mediana_3rot':float(rt.distancia_outros_mediana.median()),
                 'menor_grupo':int(np.bincount(y,minlength=4).min())})
            for idg,g in mapping.items():
                mask=y==idg
                for p in PLATES:
                    pp=df[P].to_numpy()==p
                    perplate_rows.append({'variante':variant,'grupo':g,'placa':p,'n':int(np.sum(mask&pp)),
                                          'n_placa':int(pp.sum()),'pct_na_placa':float(100*np.mean(mask[pp]))})
                for j,b in enumerate(CHANNELS):
                    v=x[mask,j];a=inp['A'][mask,:,j]
                    profile_rows.append({'variante':variant,'grupo':g,'canal':b,'n':int(mask.sum()),
                             'mediana_contraste':float(np.median(v)),'q25_contraste':float(np.quantile(v,.25)),
                             'q75_contraste':float(np.quantile(v,.75)),
                             'mediana_A_original':float(np.median(np.median(a,axis=1)))})
            print(variant+': tres rotacoes e recuperacao das mesmas condicoes concluidas.',flush=True)
        # Repeated DMSO holdouts: primary model uses 28 controls; sensitivity uses 14.
        split_rows=[];hold_rows=[];role_rows=[]
        for s in range(CFG['n_splits']):
            selection=split_controls(CFG['seed']+40000+s);half_mu=center_controls(dc,selection)
            holds=heldout_controls(dc,selection,half_mu)
            for row in holds:row['divisao']=s+1
            hold_rows.extend(holds)
            for ip,p in enumerate(PLATES):
                for ir,r in enumerate(REPS):
                    for iw,w in enumerate(wells[ip,ir]):
                        role_rows.append({'divisao':s+1,'placa':p,'replica':r,'poco':w,
                                          'papel':'calibracao' if selection[ip,ir,iw] else 'avaliacao'})
            for ir in range(3):
                row,detail,_=evaluate_rotation(inp,half_mu,ir,seed+20000+ir,True)
                row.update({'divisao':s+1,'n_DMSO_centro':14});split_rows.append(row)
                detail['divisao']=s+1;detail['replica_aplicacao']=REPS[ir]
                # Keep per-condition main retrieval only; split statistics suffice for sensitivity.
            if (s+1)%5==0:print(f'Divisoes de DMSO: {s+1}/{CFG["n_splits"]} concluidas.',flush=True)
    # Mechanical check: a shared translation must preserve within-batch distances.
    invariance=[]
    for ip,p in enumerate(PLATES):
        inds=np.flatnonzero(idx==ip)
        for ir,r in enumerate(REPS):
            u=aggregate_profile(inp['A'],idx,[ir])[inds]
            v=aggregate_profile(inp['A'],idx,[ir],mu)[inds]
            dd=np.max(np.abs(pdist(u)-pdist(v)))
            check(dd<1e-10,'Distancias intraplaca alteradas por traducao.')
            invariance.append({'placa':p,'replica':r,'n_condicoes':len(inds),
                               'max_erro_distancia_intrabatch':float(dd)})
    res={'resumo':pd.DataFrame(summaries),'rotacoes':pd.DataFrame(rotation_rows),
         'recuperacao_por_condicao':pd.concat(retrieval_rows,ignore_index=True),
         'atribuicoes':assign,'centros_DMSO':pd.DataFrame(murows),
         'grupos_por_placa':pd.DataFrame(perplate_rows),'perfis_grupos':pd.DataFrame(profile_rows),
         'divisoes_rotacoes':pd.DataFrame(split_rows),'DMSO_avaliacao':pd.DataFrame(hold_rows),
         'divisoes_papeis_controles':pd.DataFrame(role_rows),'invariancia_intraplaca':pd.DataFrame(invariance),
         'conferencia_originais':inp['conferences']}
    res['localizacao_12_prioridades']=assign.loc[assign.prioridade_R1_R3_original].copy()
    totals=res['recuperacao_por_condicao'].groupby(['variante','replica_aplicacao','placa'],sort=True)
    res['recuperacao_por_placa']=totals.agg(n=('condition_index','size'),
          fracao_par_mais_proximo=('fracao_par_mais_proximo','mean'),distancia_par=('distancia_par','median'),
          distancia_outros=('distancia_outros_mediana','median')).reset_index()
    c=pd.crosstab(assign.A_original_grupo,assign.A_centrado_DMSO_grupo)
    res['cruzamento_grupos']=c.rename_axis('grupo_original').reset_index()
    a=res['resumo'].set_index('variante')
    res['mudancas_por_condicao']=assign[KEY+['prioridade_R1_R3_original']].copy()
    pairs=res['recuperacao_por_condicao'].groupby(['condition_index','variante']).fracao_par_mais_proximo.mean().unstack()
    res['mudancas_por_condicao']['delta_recuperacao']=pairs['A_centrado_DMSO'].to_numpy()-pairs['A_original'].to_numpy()
    res['resumo']['ARI_original_calibrado']=adjusted_rand_score(assign.A_original_grupo,assign.A_centrado_DMSO_grupo)
    for name,t in res.items():t.to_csv(out/'tabelas'/(name+'.csv'),index=False)
    (out/'modelos_calibracao.json').write_text(json.dumps(models,ensure_ascii=False,indent=2),encoding='utf-8')
    return res

### Figuras

In [ ]:
def figures07(res,out):
    captions={}
    def save(fig,name,caption):
        fig.tight_layout()
        fig.savefig(out/'figuras'/(name+'.png'),dpi=220,bbox_inches='tight')
        fig.savefig(out/'figuras'/(name+'.svg'),bbox_inches='tight')
        plt.close(fig);captions[name]=caption
    rt=res['rotacoes']
    fig,ax=plt.subplots(figsize=(8,4.8))
    for r in REPS:
        t=rt.query('replica_aplicacao==@r').set_index('variante').loc[VARIANTS]
        ax.plot([0,1],t.ARI,'o-',label=r+' na aplicacao')
    ax.set_xticks([0,1],[LABELS[v] for v in VARIANTS]);ax.set_ylim(-.1,1.02)
    ax.set_ylabel('ARI: ajuste em duas replicas / aplicacao na terceira')
    ax.set_title('Calibracao residual dos contrastes: k = 4 fixo');ax.legend()
    save(fig,'T01_ARI_rotacoes','Tres rotacoes das mesmas condicoes, com reajuste de escala e K-means. Os DMSO da replica aplicada calibram apenas essa replica. Nao e teste independente do pipeline completo.')
    fig,ax=plt.subplots(figsize=(8,4.8))
    for r in REPS:
        t=rt.query('replica_aplicacao==@r').set_index('variante').loc[VARIANTS]
        ax.plot([0,1],100*t.fracao_par_mais_proximo_media,'o-',label=r+' na aplicacao')
    ax.axhline(50,linestyle=':',label='50%: referencia de ordenacao, nao teste')
    ax.set_xticks([0,1],[LABELS[v] for v in VARIANTS]);ax.set_ylim(0,100)
    ax.set_ylabel('Outros compostos mais distantes que a replica correta (%)')
    ax.set_title('Preservacao da correspondencia entre condicoes');ax.legend(loc='lower left')
    save(fig,'T02_recuperacao_condicoes','Distancias em metrica comum: IQR da representacao original ajustado nas duas replicas de treinamento, usado antes e depois da correcao. Comparacoes dentro da mesma placa, excluindo o mesmo composto dos negativos; empates recebem metade do credito. Nao e acuracia de classificador nem validacao biologica.')
    h=res['DMSO_avaliacao'].groupby(['divisao','variante']).first().reset_index()
    fig,ax=plt.subplots(figsize=(8,4.8))
    for variant in VARIANTS:
        t=h.query('variante==@variant')
        ax.plot(t.divisao,t.dispersao_medias_21_grupos,'o-',label=LABELS[variant],markersize=4)
    ax.set_xlabel('Divisao aleatoria dos controles (14 ajuste / 14 avaliacao)')
    ax.set_ylabel('Dispersao das medias DMSO entre 21 grupos')
    ax.set_title('Avaliacao em controles nao usados no ajuste adicional');ax.legend()
    save(fig,'T03_DMSO_fora_ajuste','Raiz da media dos quadrados das diferencas entre os centros dos 21 grupos placa x replica e seu centro global. Cada ponto usa apenas os 14 controles de avaliacao por grupo. As 20 divisoes se sobrepoem e nao sao experimentos independentes. MAD/P95 anteriores usaram todos os controles.')
    mito=res['DMSO_avaliacao'].query('canal=="Mito"').groupby(['divisao','variante','placa']).media_controles_avaliacao.mean().reset_index()
    fig,ax=plt.subplots(figsize=(9,4.8))
    for iv,variant in enumerate(VARIANTS):
        x=np.arange(7)+(iv-.5)*.15
        t=mito.query('variante==@variant').groupby('placa').media_controles_avaliacao.agg(['mean','min','max']).loc[PLATES]
        ax.errorbar(x,t['mean'],yerr=np.vstack([t['mean']-t['min'],t['max']-t['mean']]),fmt='o',capsize=3,label=LABELS[variant])
    ax.axhline(0,linestyle=':');ax.set_xticks(np.arange(7),PLATES);ax.set_ylabel('Media C_Mito dos controles de avaliacao')
    ax.set_title('Centro residual de Mito por placa: medias das replicas');ax.legend()
    save(fig,'T04_DMSO_Mito_por_placa','Cada divisao resume suas tres replicas com peso igual. Pontos: media das 20 divisoes; barras: minimo e maximo, nao intervalos de confianca. Os controles exibidos nao estimaram a correcao adicional de sua propria divisao.')
    fig,ax=plt.subplots(figsize=(7.5,4.8));s=res['resumo'].set_index('variante').loc[VARIANTS]
    bars=ax.bar([LABELS[v] for v in VARIANTS],s.AMI_placa)
    for bar,val in zip(bars,s.AMI_placa):ax.annotate(f'{val:.4f}',(bar.get_x()+bar.get_width()/2,bar.get_height()),xytext=(0,5),textcoords='offset points',ha='center')
    ax.set_ylim(0,max(s.AMI_placa)*1.25+.01);ax.set_ylabel('Informacao mutua ajustada (AMI)');ax.set_title('Associacao entre grupos de tratamentos e placas')
    save(fig,'T05_associacao_placa','AMI nao e uma porcentagem de efeito tecnico. Menor associacao com placa nao demonstra maior preservacao de biologia nem deve ser usada isoladamente para selecionar a representacao.')
    fig,ax=plt.subplots(figsize=(9,4.8));profiles=res['perfis_grupos'].query('variante=="A_centrado_DMSO"')
    for g in sorted(profiles.grupo.unique()):
        t=profiles.query('grupo==@g').set_index('canal').loc[CHANNELS]
        ax.plot(np.arange(4),t.mediana_contraste,'o-',label=f'{g} (n={int(t.n.iloc[0])})')
    ax.set_xticks(np.arange(4),CHANNELS);ax.axhline(0,linestyle=':');ax.set_ylabel('Mediana do contraste com centro DMSO descontado')
    ax.set_title('Perfis dos novos grupos: escala assinada, sem corte A = 1');ax.legend()
    save(fig,'T06_perfis_calibrados','Medianas por canal na representacao ajustada das condicoes. Medianas marginais nao precisam somar zero. J01-J04 sao identificadores de novos grupos, nao classes biologicas nem equivalentes automaticos a G01-G04. Zero e uma origem de referencia, nao limiar de significancia.')
    t=res['grupos_por_placa'].query('variante=="A_centrado_DMSO"');fig,ax=plt.subplots(figsize=(10,5))
    for i,g in enumerate(sorted(t.grupo.unique())):
        v=t.query('grupo==@g').set_index('placa').loc[PLATES]
        ax.bar(np.arange(7)+(i-1.5)*.2,v.pct_na_placa,width=.19,label=g)
    ax.set_xticks(np.arange(7),PLATES);ax.set_ylabel('Condicoes no grupo / total da placa (%)');ax.set_ylim(0,100)
    ax.set_title('Composicao dos novos grupos por placa');ax.legend(ncol=4)
    save(fig,'T07_composicao_placas','Denominador: total de condicoes completas na respectiva placa. Composicao quimica e componentes tecnicos nao sao identificados separadamente.')
    split=res['divisoes_rotacoes'].groupby('divisao').ARI.median()
    fig,ax=plt.subplots(figsize=(9,4.8));ax.plot(split.index,split.values,'o-',label='Calibracao com 14 DMSO: mediana das 3 rotacoes')
    ax.axhline(s.loc['A_original','ARI_rotacoes_mediana'],linestyle=':',label='Original: mediana das 3 rotacoes')
    ax.axhline(s.loc['A_centrado_DMSO','ARI_rotacoes_mediana'],linestyle='--',label='Calibracao com 28 DMSO: mediana das 3 rotacoes')
    ax.set_ylim(-.1,1.02);ax.set_xlabel('Divisao aleatoria dos DMSO');ax.set_ylabel('ARI mediano');ax.set_title('Sensibilidade aos controles usados na calibracao');ax.legend(loc='best')
    save(fig,'T08_sensibilidade_controles','Vinte calibracoes com metade dos DMSO; k e sementes do K-means permanecem iguais. As divisoes estimam sensibilidade ao conjunto de controles, nao a incerteza de novos experimentos. Nenhuma divisao foi escolhida para substituir o ajuste principal.')
    (out/'legendas.md').write_text('\n\n'.join('## '+n+'\n'+c for n,c in captions.items()),encoding='utf-8')
    return captions

### Relatorio, texto do TCC e exportacao

In [ ]:
def reports07(res,out,execution):
    sm=res['resumo'].set_index('variante');a=sm.loc['A_original'];b=sm.loc['A_centrado_DMSO']
    h=res['DMSO_avaliacao'].groupby(['divisao','variante']).first().reset_index()
    hb=h.query('variante=="A_original"').dispersao_medias_21_grupos.median()
    hc=h.query('variante=="A_centrado_DMSO"').dispersao_medias_21_grupos.median()
    split=res['divisoes_rotacoes'].groupby('divisao').ARI.median()
    r=res['rotacoes'].pivot(index='replica_aplicacao',columns='variante',values='ARI').loc[REPS]
    rr=res['rotacoes'].pivot(index='replica_aplicacao',columns='variante',values='fracao_par_mais_proximo_media').loc[REPS]
    improved_ari=int((r.A_centrado_DMSO>r.A_original).sum());improved_retrieval=int((rr.A_centrado_DMSO>rr.A_original).sum())
    deltas=res['mudancas_por_condicao'].delta_recuperacao
    text=f'''# Calibracao residual dos contrastes por DMSO - etapa 07

Execucao: {execution}. Analise retrospectiva de desenvolvimento. 2447 condicoes, 588 DMSO, R1-R3; k=4 fixo.
Comparador: contrastes de A originais. Nova representacao: o centro medio dos DMSO de placa x replica e descontado. Nenhuma escala DMSO adicional, whitening ou busca de hiperparametros.

## Agregacao e fronteira entre ajuste e aplicacao
Em uma replica, C_ajustado = C(A) - mu_DMSO. Em duas ou tres replicas, foi preservada a agregacao original C(mediana de A), da qual se subtraiu a media dos centros DMSO das replicas incluidas. Isso NAO e mediana(C-mu); a ordem foi explicitada para nao mudar silenciosamente o comparador. A correcao nula reproduz exatamente a entrada original.
Na rotacao, as duas replicas de ajuste usam somente seus centros; os tratamentos da terceira replica nao estimam nem o centro DMSO nem a escala nem o K-means. Os DMSO da terceira replica calibram sua aplicacao, como parte do procedimento declarado. O k e as etapas anteriores ja foram definidos em R1-R3 completos.

## Resultados principais (28 controles para cada centro)
| Metrica | Original | Calibrado por DMSO |
|---|---:|---:|
| ARI mediano nas rotacoes | {a.ARI_rotacoes_mediana:.4f} | {b.ARI_rotacoes_mediana:.4f} |
| Mesmo grupo nas tres projecoes pelo modelo fixo (%) | {a.pct_3rep_mesmo_grupo:.2f} | {b.pct_3rep_mesmo_grupo:.2f} |
| AMI com placa | {a.AMI_placa:.4f} | {b.AMI_placa:.4f} |
| Fracao media de outros compostos mais distantes que a replica correta (%) | {100*a.fracao_par_mais_proximo_media_3rot:.2f} | {100*b.fracao_par_mais_proximo_media_3rot:.2f} |
| Mediana das distancias ao par: mediana das tres rotacoes | {a.distancia_par_mediana_3rot:.4f} | {b.distancia_par_mediana_3rot:.4f} |
| Mediana das distancias a outros compostos: mediana das tres rotacoes | {a.distancia_outros_mediana_3rot:.4f} | {b.distancia_outros_mediana_3rot:.4f} |
| Menor grupo de referencia | {int(a.menor_grupo)} | {int(b.menor_grupo)} |

ARI original versus nova particao: {b.ARI_original_calibrado:.4f}.
Aumentos numericos de ARI em {improved_ari}/3 rotacoes; aumentos de recuperacao em {improved_retrieval}/3. Nao sao testes de significancia.
'''
    for rname in REPS:
        text+=f'\n{rname}: ARI {r.loc[rname,"A_original"]:.4f} -> {r.loc[rname,"A_centrado_DMSO"]:.4f}; recuperacao {100*rr.loc[rname,"A_original"]:.2f}% -> {100*rr.loc[rname,"A_centrado_DMSO"]:.2f}%.\n'
    text+=f'''
## Controles fora do ajuste adicional
Foram realizadas 20 divisoes aleatorias, com 14 DMSO de calibracao e 14 de avaliacao por placa x replica. Dispersao mediana das medias dos 21 grupos, calculada nos controles de avaliacao: {hb:.6f} -> {hc:.6f}.
ARI mediano das tres rotacoes por divisao de controles: mediana {split.median():.4f}, minimo {split.min():.4f}, maximo {split.max():.4f}. Essas faixas NAO sao intervalos de confianca; as divisoes se sobrepoem.
Todos os controles ja participaram de MAD/P95 anteriores; esta separacao avalia apenas a correcao residual adicional. Centrar os 28 controles usados para o proprio ajuste em zero e uma identidade, nao uma validacao.

## Preservacao da geometria e discriminacao
As distancias entre tratamentos dentro da mesma placa e replica foram preservadas antes do reajuste global da escala, com erro numerico maximo {res['invariancia_intraplaca'].max_erro_distancia_intrabatch.max():.3g}.
A recuperacao compara a mesma condicao a outros compostos da MESMA placa, com metrica comum antes/depois (IQR original aprendido nas duas replicas de ajuste). Empates contam metade; 50% e uma referencia de ordenacao, nao valor-p, FDR ou acuracia.
Delta medio de recuperacao por condicao: mediana {deltas.median():+.4f}; {int((deltas>0).sum())} condicoes aumentaram e {int((deltas<0).sum())} diminuiram numericamente. Nao foram selecionadas apenas as condicoes com melhora.
A correspondencia entre replicas pode incluir efeitos de posicao dos pocos ou outras variacoes compartilhadas. Identidade da condicao nao e rotulo de mecanismo de acao. Preservacao de distancias e uma propriedade matematica; nao prova preservacao de toda informacao biologica.

## Limites e decisao
Menor AMI com placa, alinhamento dos DMSO ou maior ARI, isoladamente, nao demonstram recuperacao de biologia. A calibracao por DMSO pressupoe uma componente aditiva compartilhada com os tratamentos na escala dos contrastes; nao corrige automaticamente variancias, covariancias, caudas ou efeitos dependentes do tratamento.
A media foi escolhida antes desta execucao, sem selecionar centros/limiares apos observar os resultados. Seu risco de sensibilidade aos controles foi examinado pelas divisoes.
Os grupos J sao novos ajustes exploratorios, nao classes mitocondriais confirmadas. Nao se aplica A>1 aos contrastes ajustados. Os 12 candidatos e os resultados R4 permanecem preservados; R4 nao foi lida.
Nenhum classificador supervisionado foi treinado. Esta etapa nao e validacao independente do pipeline completo nem um novo teste confirmatorio.
'''
    (out/'relatorio_calibracao_DMSO.md').write_text(text,encoding='utf-8')
    draft=f'''Calibracao residual dos contrastes por controles DMSO
Foi investigada uma correcao de localizacao baseada exclusivamente nos controles DMSO de cada placa e replica. O vetor medio de contrastes dos 28 controles foi subtraido da representacao dos tratamentos. Nas analises com duas ou tres replicas, preservou-se a representacao original C(mediana de A), descontando a media dos centros de controle das replicas incluidas. Os numeros de grupos e os parametros do K-means foram mantidos, com reajuste apenas da escala robusta global e dos centros na nova entrada. A mesma correcao foi avaliada em 20 divisoes com 14 controles de calibracao e 14 de avaliacao por placa e replica, condicionadas a normalizacao e as referencias P95 anteriores.
O ARI mediano nas rotacoes entre replicas passou de {a.ARI_rotacoes_mediana:.4f} para {b.ARI_rotacoes_mediana:.4f}; a associacao dos grupos com placas, medida por AMI, passou de {a.AMI_placa:.4f} para {b.AMI_placa:.4f}. No modelo fixo, a proporcao de condicoes com atribuicao comum nas tres replicas foi de {a.pct_3rep_mesmo_grupo:.2f}% e {b.pct_3rep_mesmo_grupo:.2f}%, respectivamente. O diagnostico continuo de correspondencia entre condicoes, medido pela fracao de outros compostos da mesma placa mais distantes que o par correto, apresentou media de {100*a.fracao_par_mais_proximo_media_3rot:.2f}% antes e {100*b.fracao_par_mais_proximo_media_3rot:.2f}% depois da correcao. A escala das distancias foi mantida comum nas comparacoes antes/depois.
Nos controles reservados somente para a avaliacao da correcao adicional, a dispersao mediana dos centros dos 21 grupos placa x replica passou de {hb:.6f} para {hc:.6f}. As divisoes reutilizam observacoes e nao constituem experimentos independentes. Os resultados foram tratados como avaliacao retrospectiva de uma nova representacao, sem identificar mecanismos biologicos, recalibrar o criterio original de prioridade ou utilizar R4.\n'''
    (out/'texto_TCC_calibracao_DMSO.md').write_text(draft,encoding='utf-8')
    return text


def run07(root,out=None):
    """Run with immutable inputs and a fresh output directory. Never opens R4."""
    execution=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    if out is None:out=Path(root)/'resultados_tcc/calibracao_residual_DMSO'/('R1_R3_'+execution)
    out=Path(out);check(not out.exists(),'Pasta de saida ja existe; use uma nova para preservar o historico.')
    out.mkdir(parents=True);(out/'tabelas').mkdir();(out/'figuras').mkdir()
    plan={'execution_utc':execution,'config':CFG,'sources':SOURCES,
          'question':'A calibracao residual por DMSO melhora concordancia entre replicas sem colapsar contrastes entre tratamentos?',
          'prior_decisions':'k=4; media de C_A nos DMSO; uma traducao por placa x replica; comparador A original; sem novos metodos',
          'scope':'Nova analise retrospectiva. R4 nunca lida. Etapas anteriores fixas.',
          'aggregation':'Manter C(mediana A) e subtrair a media dos centros DMSO das replicas incluidas; nao usar mediana(C-mu).'}
    (out/'plano_antes_ajuste.json').write_text(json.dumps(plan,indent=2,ensure_ascii=False),encoding='utf-8')
    inp=load(root)
    res=analyse_calibration(inp,out);caps=figures07(res,out);report=reports07(res,out,execution)
    for name,path in inp['paths'].items():check(sha256(path)==SOURCES[name]['sha256'],'Entrada alterada: '+name)
    manifest={'execution':execution,'code_version':CFG['protocol_version'],'config':CFG,
              'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
                          'scikit-learn':sklearn.__version__},'readers':inp['readers'],
              'input_hashes_unchanged':{n:sha256(p) for n,p in inp['paths'].items()},
              'outputs':{str(p.relative_to(out)):sha256(p) for p in out.rglob('*') if p.is_file()},
              'R4_read':False,'figure_count':len(caps),'n_full_controls':588,'n_conditions':2447}
    (out/'manifesto_calibracao.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
    zip_path=Path(shutil.make_archive(str(out),'zip',root_dir=out))
    print('CONCLUIDO:',len(caps),'figuras; 3 rotacoes; 20 divisoes 14/14 de DMSO.')
    print('Pasta:',out,'\nPacote:',zip_path)
    print('R4 nao utilizada. Nenhum limiar ou candidato original alterado.')
    return out,res,report

## Executar a comparacao

In [ ]:
# Verificacoes rapidas antes da leitura dos resultados reais.
rng = np.random.default_rng(123)
synthetic = rng.normal(size=(7,3,28,4))
synthetic -= synthetic.mean(-1, keepdims=True)
sel = split_controls(1)
assert (sel.sum(2)==14).all() and not (sel & ~sel).any()
mu_test = center_controls(synthetic)
np.testing.assert_allclose(mu_test.sum(-1), 0, atol=1e-12)
check_data = np.exp(rng.normal(size=(14,3,4)))
plate_index_test = np.tile(np.arange(7),2)
for rr in ([0],[0,1],[0,1,2]):
    np.testing.assert_array_equal(aggregate_profile(check_data,plate_index_test,rr),
        aggregate_profile(check_data,plate_index_test,rr,np.zeros((7,3,4))))
del synthetic, check_data, mu_test, sel
print('Verificacoes de separacao, soma zero e comparador nulo: OK.')


In [ ]:
SAIDA, RESULTADOS, RELATORIO = run07(RAIZ)
display(RESULTADOS["resumo"])
display(RESULTADOS["rotacoes"])
display(Markdown(RELATORIO))

## Figuras da sua execucao

In [ ]:
for arquivo in sorted((SAIDA / "figuras").glob("*.png")):
    display(Markdown("### " + arquivo.stem))
    display(Image(filename=str(arquivo), width=950))
display(Markdown((SAIDA / "legendas.md").read_text(encoding="utf-8")))

## Texto calculado para incorporacao ao TCC

In [ ]:
display(Markdown((SAIDA / "texto_TCC_calibracao_DMSO.md").read_text(encoding="utf-8")))
print("CONCLUIDO. Pasta:", SAIDA)
print("R4 nao utilizada; candidatos e escores anteriores preservados.")

## Referencias de apoio (nao sao fontes dos resultados numericos)

Arevalo et al. (2024), *Evaluating batch correction methods for image-based cell profiling*. Nature Communications 15, 6516. https://doi.org/10.1038/s41467-024-50613-5. Fundamenta a necessidade de avaliar correspondencia das perturbacoes juntamente com variacao de lote; a traducao de contrastes deste notebook e uma proposta propria, nao uma reproducao do metodo desse artigo.

K-means: https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html

Escala robusta: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html

ARI: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.adjusted_rand_score.html

Nao selecionar outro centro, algoritmo, limiar ou k com base no resultado desta execucao sem registrar uma nova pergunta. ARI/AMI, silhouette e recuperacao nao demonstram mecanismo de acao ou seletividade biologica.
